# Proyecto 1 - Scraping MINEDUC

## Objetivo

Obtener los establecimientos educativos con nivel Diversificado desde el portal del MINEDUC utilizando Web Scraping.

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

### 1. Conexión con el sitio

In [2]:
import requests

URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/wbfBuscar.aspx"

session = requests.Session()

response = session.get(URL)

print(response.status_code)

200


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>
	Búsqueda de centros educativos autorizados por el Mineduc
</title>


In [4]:
hidden_inputs = soup.find_all("input", type="hidden")

print(f"Hay {len(hidden_inputs)} campos ocultos")

for campo in hidden_inputs:
    print(campo.get("name"))

Hay 3 campos ocultos
__VIEWSTATE
__VIEWSTATEGENERATOR
__EVENTVALIDATION


### 2. Inspección del formulario

In [5]:
viewstate = soup.find("input", {"name": "__VIEWSTATE"})["value"]

eventvalidation = soup.find(
    "input",
    {"name": "__EVENTVALIDATION"}
)["value"]

viewstategenerator = soup.find(
    "input",
    {"name": "__VIEWSTATEGENERATOR"}
)["value"]

print(viewstate[:100])
print(eventvalidation[:100])
print(viewstategenerator)

/wEPDwUKLTk4NjAyODgxMQ9kFgJmD2QWAgIDD2QWBgINDw8WAh4EVGV4dAVaQsO6c3F1ZWRhIGRlIGNlbnRyb3MgZWR1Y2F0aXZv
/wEdADt/0dwJyZlIw8JUU+HJUNeqU90iX8s3bIMCbHwt7Ocp/pgXezQWBJR3tT/xYsBT9n1BvvCe0vdrwwW4xxLkkt0TvW2uQ10j
EDAC6F9D


### 3. Obtención de parámetros

In [6]:
for select in soup.find_all("select"):
    print(select.get("name"))

_ctl0:ContentPlaceHolder1:cmbDepartamento
_ctl0:ContentPlaceHolder1:cmbMunicipio
_ctl0:ContentPlaceHolder1:cmbNivel
_ctl0:ContentPlaceHolder1:cmbSector
_ctl0:ContentPlaceHolder1:ddlplan
_ctl0:ContentPlaceHolder1:ddlModalidad


### 4. Envío de la búsqueda

In [7]:
departamento = soup.find(
    "select",
    {"name": "_ctl0:ContentPlaceHolder1:cmbDepartamento"}
)

for option in departamento.find_all("option"):
    print(option.get("value"), "->", option.text)

SELECCIONE UNO -> SELECCIONE UNO
16 -> ALTA VERAPAZ
15 -> BAJA VERAPAZ
04 -> CHIMALTENANGO
20 -> CHIQUIMULA
00 -> CIUDAD CAPITAL
02 -> EL PROGRESO
05 -> ESCUINTLA
01 -> GUATEMALA
13 -> HUEHUETENANGO
18 -> IZABAL
21 -> JALAPA
22 -> JUTIAPA
17 -> PETEN
09 -> QUETZALTENANGO
14 -> QUICHE
11 -> RETALHULEU
03 -> SACATEPEQUEZ
12 -> SAN MARCOS
06 -> SANTA ROSA
07 -> SOLOLA
10 -> SUCHITEPEQUEZ
08 -> TOTONICAPAN
19 -> ZACAPA


In [8]:
nivel = soup.find(
    "select",
    {"name": "_ctl0:ContentPlaceHolder1:cmbNivel"}
)

for option in nivel.find_all("option"):
    print(option.get("value"), "->", option.text)

TODOS -> TODOS
45 -> BASICO
46 -> DIVERSIFICADO
42 -> PARVULOS
41 -> PREPRIMARIA BILINGUE
43 -> PRIMARIA
44 -> PRIMARIA DE ADULTOS


In [9]:
for boton in soup.find_all("input"):

    tipo = boton.get("type")

    if tipo in ["submit", "image", "button"]:

        print("---------------------")
        print("Nombre:", boton.get("name"))
        print("Tipo:", tipo)
        print("Valor:", boton.get("value"))

---------------------
Nombre: _ctl0:ContentPlaceHolder1:IbtnConsultar
Tipo: image
Valor: None
---------------------
Nombre: _ctl0:ContentPlaceHolder1:btnLimpiar
Tipo: image
Valor: None
---------------------
Nombre: _ctl0:btnCerrarAvisoLegal
Tipo: submit
Valor: Cerrar Ventana
---------------------
Nombre: _ctl0:btnCerrarTerminos
Tipo: submit
Valor: Cerrar Ventana
---------------------
Nombre: _ctl0:btnCerrarContacto
Tipo: submit
Valor: Cerrar Ventana


In [11]:
hidden_fields = {}

for campo in hidden_inputs:
    hidden_fields[campo["name"]] = campo["value"]

hidden_fields.keys()

dict_keys(['__VIEWSTATE', '__VIEWSTATEGENERATOR', '__EVENTVALIDATION'])

In [28]:
payload = form_fields.copy()

# Campos ocultos adicionales que envía ASP.NET
payload["__EVENTTARGET"] = ""
payload["__EVENTARGUMENT"] = ""
payload["__LASTFOCUS"] = ""

# Filtros
payload["_ctl0:ContentPlaceHolder1:cmbDepartamento"] = "16"
payload["_ctl0:ContentPlaceHolder1:cmbMunicipio"] = "TODOS"
payload["_ctl0:ContentPlaceHolder1:cmbNivel"] = "46"
payload["_ctl0:ContentPlaceHolder1:cmbSector"] = "TODOS"
payload["_ctl0:ContentPlaceHolder1:ddlplan"] = "TODOS"
payload["_ctl0:ContentPlaceHolder1:ddlModalidad"] = "TODOS"

# Botón Buscar
payload["_ctl0:ContentPlaceHolder1:IbtnConsultar.x"] = "80"
payload["_ctl0:ContentPlaceHolder1:IbtnConsultar.y"] = "22"

In [21]:
form_fields = {}

for inp in soup.find_all("input"):
    name = inp.get("name")

    if name:
        form_fields[name] = inp.get("value", "")

print(form_fields.keys())

dict_keys(['__VIEWSTATE', '__VIEWSTATEGENERATOR', '__EVENTVALIDATION', '_ctl0:ContentPlaceHolder1:txtCodEstab', '_ctl0:ContentPlaceHolder1:txtNomEstab', '_ctl0:ContentPlaceHolder1:txtDirecEstab', '_ctl0:ContentPlaceHolder1:IbtnConsultar', '_ctl0:ContentPlaceHolder1:btnLimpiar', '_ctl0:btnCerrarAvisoLegal', '_ctl0:btnCerrarTerminos', '_ctl0:btnCerrarContacto'])


In [23]:
form = soup.find("form")

print("Action:", form.get("action"))
print("Method:", form.get("method"))

Action: ./wbfBuscar.aspx
Method: post


In [24]:
for select in soup.find_all("select"):
    print(select.get("name"), "->", select.get("id"))

_ctl0:ContentPlaceHolder1:cmbDepartamento -> _ctl0_ContentPlaceHolder1_cmbDepartamento
_ctl0:ContentPlaceHolder1:cmbMunicipio -> _ctl0_ContentPlaceHolder1_cmbMunicipio
_ctl0:ContentPlaceHolder1:cmbNivel -> _ctl0_ContentPlaceHolder1_cmbNivel
_ctl0:ContentPlaceHolder1:cmbSector -> _ctl0_ContentPlaceHolder1_cmbSector
_ctl0:ContentPlaceHolder1:ddlplan -> _ctl0_ContentPlaceHolder1_ddlplan
_ctl0:ContentPlaceHolder1:ddlModalidad -> _ctl0_ContentPlaceHolder1_ddlModalidad


In [25]:
for select in soup.find_all("select"):
    print("\n========================")
    print(select.get("name"))

    for option in select.find_all("option"):
        print(f"{option.get('value')} -> {option.text}")


_ctl0:ContentPlaceHolder1:cmbDepartamento
SELECCIONE UNO -> SELECCIONE UNO
16 -> ALTA VERAPAZ
15 -> BAJA VERAPAZ
04 -> CHIMALTENANGO
20 -> CHIQUIMULA
00 -> CIUDAD CAPITAL
02 -> EL PROGRESO
05 -> ESCUINTLA
01 -> GUATEMALA
13 -> HUEHUETENANGO
18 -> IZABAL
21 -> JALAPA
22 -> JUTIAPA
17 -> PETEN
09 -> QUETZALTENANGO
14 -> QUICHE
11 -> RETALHULEU
03 -> SACATEPEQUEZ
12 -> SAN MARCOS
06 -> SANTA ROSA
07 -> SOLOLA
10 -> SUCHITEPEQUEZ
08 -> TOTONICAPAN
19 -> ZACAPA

_ctl0:ContentPlaceHolder1:cmbMunicipio

_ctl0:ContentPlaceHolder1:cmbNivel
TODOS -> TODOS
45 -> BASICO
46 -> DIVERSIFICADO
42 -> PARVULOS
41 -> PREPRIMARIA BILINGUE
43 -> PRIMARIA
44 -> PRIMARIA DE ADULTOS

_ctl0:ContentPlaceHolder1:cmbSector
TODOS -> TODOS
21 -> OFICIAL
22 -> PRIVADO
23 -> MUNICIPAL
24 -> COOPERATIVA

_ctl0:ContentPlaceHolder1:ddlplan
TODOS -> TODOS
1 -> DIARIO(REGULAR)
2 -> SABATINO
3 -> DOMINICAL
4 -> FIN DE SEMANA
5 -> IRREGULAR
6 -> MIXTO
8 -> A DISTANCIA

_ctl0:ContentPlaceHolder1:ddlModalidad
TODOS -> TODOS


### Automatización con Selenium

In [33]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [34]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

driver.maximize_window()

driver.get(
    "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/wbfBuscar.aspx"
)

In [35]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

print("Página cargada")

Página cargada


In [36]:
departamento = Select(
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )
)

departamento.select_by_value("16")

In [38]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbDepartamento")
    )
)

departamento = Select(
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )
)

print(departamento.first_selected_option.text)

ALTA VERAPAZ


In [39]:
# Esperamos a que el combo de nivel esté disponible
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbNivel")
    )
)

nivel = Select(
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbNivel"
    )
)

nivel.select_by_value("46")

In [40]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.NAME, "_ctl0:ContentPlaceHolder1:cmbNivel")
    )
)

nivel = Select(
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbNivel"
    )
)

print(nivel.first_selected_option.text)

DIVERSIFICADO


In [41]:
boton_buscar = driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:IbtnConsultar"
)

boton_buscar.click()

In [42]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (
            By.ID,
            "_ctl0_ContentPlaceHolder1_dgResultado"
        )
    )
)

print("Tabla encontrada")

Tabla encontrada


##### extraer la tabla 

In [43]:
from io import StringIO

import pandas as pd

html = driver.page_source

tablas = pd.read_html(StringIO(html))

print(f"Se encontraron {len(tablas)} tablas")

Se encontraron 10 tablas


In [44]:
for i, tabla in enumerate(tablas):
    print(f"\nTabla {i}")
    print(tabla.head())


Tabla 0
    0                                                  1                  2  \
0 NaN                                  [  e-Servicios  ]  [  e-Servicios  ]   
1 NaN  Búsqueda de centros educativos autorizados por...                NaN   
2 NaN                                                NaN                NaN   
3 NaN                                                NaN                NaN   
4 NaN                                                NaN                NaN   

                   3   4   5  
0  [  e-Servicios  ] NaN NaN  
1                NaN NaN NaN  
2                NaN NaN NaN  
3                NaN NaN NaN  
4                NaN NaN NaN  

Tabla 1
    0   1   2   3
0 NaN NaN NaN NaN

Tabla 2
    0   1   2
0 NaN NaN NaN

Tabla 3
                                                  0   \
0  Importante:  Para una mejor búsqueda puede sel...   
1  Importante:  Para una mejor búsqueda puede sel...   
2                                                NaN   
3  Departamento

##### encontrar tabla correcta 


In [52]:
df = tablas[9].copy()

# La primera fila contiene los nombres reales
df.columns = df.iloc[0]

# Eliminamos esa primera fila
df = df.iloc[1:].reset_index(drop=True)

df.head()

,NaN,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,NaN,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,NaN,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,NaN,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,NaN,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,NaN,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 476 entries, 0 to 475
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   nan              0 non-null      float64
 1   CODIGO           475 non-null    str    
 2   DISTRITO         466 non-null    str    
 3   DEPARTAMENTO     475 non-null    str    
 4   MUNICIPIO        475 non-null    str    
 5   ESTABLECIMIENTO  474 non-null    str    
 6   DIRECCION        473 non-null    str    
 7   TELEFONO         450 non-null    str    
 8   SUPERVISOR       466 non-null    str    
 9   DIRECTOR         415 non-null    str    
 10  NIVEL            475 non-null    str    
 11  SECTOR           475 non-null    str    
 12  AREA             475 non-null    str    
 13  STATUS           475 non-null    str    
 14  MODALIDAD        475 non-null    str    
 15  JORNADA          475 non-null    str    
 16  PLAN             475 non-null    str    
 17  DEPARTAMENTAL    475 non-nu

In [54]:
df.columns.tolist()

[np.float64(nan),
 'CODIGO',
 'DISTRITO',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'ESTABLECIMIENTO',
 'DIRECCION',
 'TELEFONO',
 'SUPERVISOR',
 'DIRECTOR',
 'NIVEL',
 'SECTOR',
 'AREA',
 'STATUS',
 'MODALIDAD',
 'JORNADA',
 'PLAN',
 'DEPARTAMENTAL']

In [57]:
# Cambiar al departamento Baja Verapaz
departamento = Select(
    driver.find_element(
        By.NAME,
        "_ctl0:ContentPlaceHolder1:cmbDepartamento"
    )
)

departamento.select_by_value("15")

In [58]:
# Hacer clic en Buscar Establecimiento
boton_buscar = driver.find_element(
    By.NAME,
    "_ctl0:ContentPlaceHolder1:IbtnConsultar"
)

boton_buscar.click()

In [59]:
from io import StringIO

html = driver.page_source

tablas = pd.read_html(StringIO(html))

df = tablas[9].copy()

# Convertir la primera fila en encabezados
df.columns = df.iloc[0]

# Eliminar esa fila
df = df.iloc[1:].reset_index(drop=True)

# Eliminar la columna sin nombre
df = df.loc[:, df.columns.notna()]

In [60]:
print(df["DEPARTAMENTO"].unique())

<StringArray>
['BAJA VERAPAZ', nan]
Length: 2, dtype: str


In [61]:
df.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


##### probamos


In [62]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al path
sys.path.append(str(Path.cwd().parent))

from src.scraper import (
    iniciar_driver,
    obtener_departamento,
)

In [63]:
driver = iniciar_driver()

In [64]:
df = obtener_departamento(driver, "16")

df.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


In [65]:
df = obtener_departamento(driver, "15")

df.head()

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


#### descargamos TODOS lso departamentos

In [66]:
departamentos = {
    "16": "ALTA VERAPAZ",
    "15": "BAJA VERAPAZ",
    "04": "CHIMALTENANGO",
    "20": "CHIQUIMULA",
    "00": "CIUDAD CAPITAL",
    "02": "EL PROGRESO",
    "05": "ESCUINTLA",
    "01": "GUATEMALA",
    "13": "HUEHUETENANGO",
    "18": "IZABAL",
    "21": "JALAPA",
    "22": "JUTIAPA",
    "17": "PETEN",
    "09": "QUETZALTENANGO",
    "14": "QUICHE",
    "11": "RETALHULEU",
    "03": "SACATEPEQUEZ",
    "12": "SAN MARCOS",
    "06": "SANTA ROSA",
    "07": "SOLOLA",
    "10": "SUCHITEPEQUEZ",
    "08": "TOTONICAPAN",
    "19": "ZACAPA",
}

In [67]:
todos_los_dfs = []

In [69]:
for codigo in departamentos:

    print(f"Descargando {departamentos[codigo]}...")

    df = obtener_departamento(driver, codigo)

    df["DEPARTAMENTO_DESCARGADO"] = departamentos[codigo]

    todos_los_dfs.append(df)

    print(df.shape)

Descargando ALTA VERAPAZ...
(475, 18)
Descargando BAJA VERAPAZ...
(171, 18)
Descargando CHIMALTENANGO...
(435, 18)
Descargando CHIQUIMULA...
(239, 18)
Descargando CIUDAD CAPITAL...
(2161, 18)
Descargando EL PROGRESO...
(158, 18)
Descargando ESCUINTLA...
(708, 18)
Descargando GUATEMALA...
(1908, 18)
Descargando HUEHUETENANGO...
(591, 18)
Descargando IZABAL...
(413, 18)
Descargando JALAPA...
(183, 18)
Descargando JUTIAPA...
(392, 18)
Descargando PETEN...
(516, 18)
Descargando QUETZALTENANGO...
(551, 18)
Descargando QUICHE...
(322, 18)
Descargando RETALHULEU...
(364, 18)
Descargando SACATEPEQUEZ...
(430, 18)
Descargando SAN MARCOS...
(724, 18)
Descargando SANTA ROSA...
(213, 18)
Descargando SOLOLA...
(192, 18)
Descargando SUCHITEPEQUEZ...
(437, 18)
Descargando TOTONICAPAN...
(128, 18)
Descargando ZACAPA...
(156, 18)


In [70]:
dataset = pd.concat(todos_los_dfs, ignore_index=True)

print(dataset.shape)

dataset.head()

(12948, 18)


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,DEPARTAMENTO_DESCARGADO
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTA VERAPAZ
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTA VERAPAZ
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTA VERAPAZ
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTA VERAPAZ


In [71]:
dataset.to_csv(
    "../data/raw/establecimientos_diversificado_guatemala.csv",
    index=False,
    encoding="utf-8"
)

print("CSV guardado correctamente.")

CSV guardado correctamente.
